# Análise exploratória — taxa de desocupação (PNAD Contínua)

Este notebook responde às três perguntas de pesquisa do artigo. Execute primeiro `python -m src.etl` na raiz do repositório. A série nacional usa trimestres móveis; a série das UFs usa trimestres civis. Essas granularidades não devem ser misturadas diretamente.


In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

# Funciona tanto com o kernel iniciado na raiz quanto dentro de notebooks/.
ROOT = Path.cwd()
if not (ROOT / 'data' / 'processed').exists():
    ROOT = ROOT.parent
BR_PATH = ROOT / 'data' / 'processed' / 'taxa_desocupacao_brasil_trimestre_movel.csv.gz'
UF_PATH = ROOT / 'data' / 'processed' / 'taxa_desocupacao_uf_trimestral.csv.gz'
if not BR_PATH.exists() or not UF_PATH.exists():
    raise FileNotFoundError('Execute primeiro: python -m src.etl')

br = pd.read_csv(BR_PATH, parse_dates=['data'])
uf = pd.read_csv(UF_PATH)
print('Brasil:', br.shape, '| UFs:', uf.shape)
display(br.head(), uf.head())


## Verificação inicial e qualidade dos dados

A inspeção abaixo identifica tipos, valores ausentes, duplicidades e estatísticas descritivas. A taxa é expressa em porcentagem; diferenças entre taxas são expressas em pontos percentuais (p.p.).


In [ ]:
br.info()
display(br[['taxa', 'variacao_12m_pp', 'desvio_sazonal_pp']].describe().round(2))
print('Duplicidades Brasil (data):', br['data'].duplicated().sum())
print('Duplicidades UF (UF/ano/trimestre):', uf.duplicated(['territorio_cod', 'ano', 'trimestre']).sum())
print('Nulos Brasil:', br['taxa'].isna().sum(), '| Nulos UFs:', uf['taxa'].isna().sum())


## RQ1 — Como evoluiu a taxa nacional entre 2012 e 2026?

A série nacional é mensal em termos de mês de encerramento, mas cada observação representa um trimestre móvel. Por isso, observações consecutivas compartilham dois meses.


In [ ]:
br_valid = br.dropna(subset=['taxa']).copy()
fig = px.line(br_valid, x='data', y='taxa', labels={'taxa':'Taxa de desocupação (%)','data':'Mês de encerramento do trimestre móvel'}, title='Taxa de desocupação — Brasil')
fig.show()
maximo = br_valid.loc[br_valid['taxa'].idxmax()]
minimo = br_valid.loc[br_valid['taxa'].idxmin()]
print(f'Máxima: {maximo.taxa:.1f}% em {maximo.data:%m/%Y}')
print(f'Mínima: {minimo.taxa:.1f}% em {minimo.data:%m/%Y}')
print('Média:', round(br_valid.taxa.mean(), 2), '| Mediana:', round(br_valid.taxa.median(), 2), '| Desvio-padrão:', round(br_valid.taxa.std(), 2))


## RQ2 — Como a taxa varia entre UFs e Grandes Regiões?

Para comparar anos, são consideradas apenas combinações UF/ano com quatro trimestres válidos. A média simples das UFs não é ponderada pelo tamanho da força de trabalho e, portanto, não representa uma taxa regional oficial.


In [ ]:
anual_uf = (uf.dropna(subset=['taxa'])
    .groupby(['regiao','uf','ano'])['taxa']
    .agg(media='mean', trimestres_validos='count').reset_index())
anual_uf = anual_uf[anual_uf['trimestres_validos'] == 4]
matriz = anual_uf.pivot(index=['regiao','uf'], columns='ano', values='media')
fig = px.imshow(matriz, aspect='auto', color_continuous_scale='Blues', labels={'color':'Taxa média (%)'}, title='Média anual da taxa por UF')
fig.show()

ultimo = uf.dropna(subset=['taxa']).sort_values(['ano','trimestre'])
ultimo_periodo = ultimo[['ano','trimestre']].drop_duplicates().iloc[-1]
recorte = ultimo[(ultimo.ano == ultimo_periodo.ano) & (ultimo.trimestre == ultimo_periodo.trimestre)]
print(f"Período mais recente disponível: {int(ultimo_periodo.ano)}-T{int(ultimo_periodo.trimestre)}")
print('Maior taxa:', recorte.loc[recorte.taxa.idxmax(), ['uf','taxa']].to_dict())
print('Menor taxa:', recorte.loc[recorte.taxa.idxmin(), ['uf','taxa']].to_dict())
print('Amplitude entre UFs (p.p.):', round(recorte.taxa.max()-recorte.taxa.min(), 2))

regiao_ano = anual_uf.groupby(['ano','regiao'])['media'].mean().reset_index()
fig = px.line(regiao_ano, x='ano', y='media', color='regiao', markers=True, labels={'media':'Média simples das UFs (%)','ano':'Ano'}, title='Média simples das UFs por Grande Região')
fig.show()


## RQ3 — Existe um padrão sazonal descritivo?

O indicador `desvio_sazonal_pp` é a diferença entre a taxa de cada mês de encerramento e a média daquele ano. Ele só é calculado para anos com os 12 meses disponíveis. A exclusão de 2020 e 2021 é uma escolha exploratória para observar o padrão sem esses anos atípicos; não equivale a uma dessazonalização formal.


In [ ]:
sazonal = br[~br['ano'].isin([2020, 2021])].dropna(subset=['desvio_sazonal_pp'])
por_mes = sazonal.groupby('mes')['desvio_sazonal_pp'].agg(['mean','std','count']).reset_index()
display(por_mes.round(2))
fig = px.bar(por_mes, x='mes', y='mean', error_y='std', labels={'mean':'Desvio médio (p.p.)','mes':'Mês de encerramento'}, title='Desvio médio em relação à média anual')
fig.show()


## Síntese e limitações

A análise é descritiva. A PNAD Contínua é uma pesquisa amostral; as taxas estaduais possuem incerteza amostral. A comparação regional feita aqui é uma média simples das UFs, não ponderada pela força de trabalho. Além disso, trimestres móveis consecutivos compartilham dois meses e não são observações independentes. Portanto, os gráficos sugerem padrões, mas não demonstram causalidade nem significância estatística.
